## Статистическо локализиране на дефекти в латентното пространство без обучение

Вие работите в завод "Атлас". За всяко изделие разполагате само с предварително изчислени embedding-и (без достъп до оригиналните изображения). Някои примери съдържат **истински дефект** (локален проблем в изделието), а други са **нормални**. Данните идват в два режима (`shift_id` = 0 или 1), които променят разпределението на embedding-ите, без това да означава дефект. Вашата задача е да откривате дефекти и да бъдете устойчиви на тези разлики между режимите.

Имате "атлас" от **нормални** embedding-и за всяка категория (`atlas_id`), записан във файловете `data/atlas_{atlas_id}.pt` (тензор `(M, D)`, който е събран от много изображения, а не само от 1, не може да си направите решетка от patch-ове), както и калибрационни данни `data/calibration.pt`, които съдържат **само нормални** примери и включват `atlas_id` и `shift_id`. Примерите за оценка, с които разполагате локално, са в **`val_data/`**: това е **малък публичен набор**, предназначен единствено да проверите дали вашият метод работи и какви резултати дава ориентировъчно. Нужно е да дадете кратко описание на решението си.

**Важно:** финалното ви решение **няма да се оценява на `val_data/`**, а ще бъде тествано на **други (скрити) данни**. Затова не "настройвайте" метода си специално за `val_data/`—целта е подход, който обобщава.

За всеки пример трябва да произведете:
- **heatmap** с размер `37×37`, където по-голяма стойност означава, че имаме по-голяма вероятност в съответния пач да има аномалия;
- **alarm** (скалар), оценка дали примерът като цяло е дефектен.

**Ограничения:** позволени са само библиотеките `torch` и `numpy`. Във всяка от функциите е абсолютно забранено пазането и зареждането на каквито и да е данни от диска. Не е позволено трениране или fine-tuning. **НЕ** променяйте таговете на клетките!

**Оценяване:**
 - разделяне на нормални и дефектни примери по **image-level** (чрез `alarm`) 
 - локализация на дефекта по **pixel/patch-level** (чрез `heatmap`). 

Използват се както непрекъснати метрики (AUC), така и дискретни метрики (F1), като дискретните изискват избор на праг/калибрация. Финалната оценка на тестовия набор се сформира като $0.5(\text{AUC}_\text{img} + \text{AUC}_\text{pix} + 0.2(\text{F1}_\text{img} + \text{F1}_\text{pix}))$. На валидационния набор ще можете да оценките само на $\text{AUC}$ метриките.

**Предаване на решението:**
 - **изпълненият jupyter notebook** (**Описанието** на задачата трябва да бъде **ТУК**), a файлът трябва да бъде наименован по следния начин: `Task_3_USER_ID.ipynb` като `USER_ID` е вашият идентификационен номер.


In [ ]:
# Изтегляне на данните от Google Drive
import os
from pathlib import Path
import zipfile
import gdown

VAL_ID = "1z-aeaEwY9yp47x2UFrCzrKD7leVM5OrW"  # val_data.zip
DATA_ID = "1HyyI-0oB72E8ak1X3gQZrMgdExGlQ7oB"  # data.zip

def download_and_extract(file_id, out_zip, out_dir):
    if not Path(out_zip).exists():
        gdown.download(id=file_id, output=out_zip, quiet=False)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(out_zip, 'r') as zf:
        zf.extractall(out_dir)

download_and_extract(VAL_ID, 'val_data.zip', './')
download_and_extract(DATA_ID, 'data.zip', './')

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

DATA_DIR = Path("data")
VAL_DIR  = Path("val_data")

# Зареждаме валидация
val_manifest = VAL_DIR / "val_manifest.json"
with open(val_manifest) as f:
    manifest = json.load(f)

# Зареждаме атласите
atlases = {}
for p in sorted(DATA_DIR.glob("atlas_*.pt")):
    atlas_id = int(p.stem.split("_")[-1])
    atlases[atlas_id] = torch.load(p, map_location="cpu").float()

if not atlases:
    raise FileNotFoundError("Няма atlas_*.pt в data/.")


# Зареждаме калибрация
calib_list = torch.load(DATA_DIR / "calibration.pt", map_location="cpu")
calibration_data = []
for item in calib_list:
    atlas_id = int(item["atlas_id"])
    shift_id = int(item["shift_id"])
    X = item["X"]
    atlas = atlases[atlas_id]
    calibration_data.append(
        {"X": X, "atlas": atlas, "atlas_id": atlas_id, "shift_id": shift_id}
    )

## Визуална интуиция

Следващата графика сравнява разпределения на L2‑нормите на embedding‑ите за:
- **Бездефектен пример (категория 0)**
- **Дефектен пример (категория 0)**


In [ ]:
X_good = next(torch.load(m["file"])["X"].float() for m in manifest
              if m["atlas_id"] == 0 and not bool(torch.load(m["file"]).get("is_anomaly", False)))

bad = next(torch.load(m["file"]) for m in manifest
           if m["atlas_id"] == 0 and bool(torch.load(m["file"]).get("is_anomaly", False)))
X_bad, mask_bad = bad["X"].float(), bad.get("mask", None)

map_good = torch.norm(X_good, p=2, dim=1).view(37,37).cpu().numpy()
map_bad  = torch.norm(X_bad,  p=2, dim=1).view(37,37).cpu().numpy()
vmin, vmax = min(map_good.min(), map_bad.min()), max(map_good.max(), map_bad.max())
fig = plt.figure(figsize=(10, 4))
gs = GridSpec(1, 4, figure=fig, width_ratios=[1, 1, 0.05, 1])
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])
axc = fig.add_subplot(gs[0, 2])
axm = fig.add_subplot(gs[0, 3])
im0 = ax0.imshow(map_good, cmap="viridis", vmin=vmin, vmax=vmax)
ax0.set_title("Бездефектен (cat 0)")
ax0.axis("off")
im1 = ax1.imshow(map_bad, cmap="viridis", vmin=vmin, vmax=vmax)
ax1.set_title("Дефектен (cat 0)")
ax1.axis("off")
fig.colorbar(im1, cax=axc, label="L2 норма")
axm.imshow(mask_bad.cpu().numpy(), cmap="gray")
axm.set_title("Маска")
axm.axis("off")
plt.tight_layout()
plt.show()


## Изчисляване на heatmap

Имплементирайте функцията `compute_heatmap(X, atlas)`.

**Вход:**
- `X`: тензор с размер `(1369, D)`
- `atlas`: тензор с размер `(M, D)`

**Изход:**
- тензор с размер `(37, 37)` - пачовете, които имат аномалия трябва да имат по-висока стойност

In [ ]:
@torch.no_grad()
def compute_heatmap(X: torch.Tensor, atlas: torch.Tensor) -> torch.Tensor:
    # TODO: Реализирайте функцията тук.
    return torch.zeros(37, 37)


## Изглаждане (smoothing)

Имплементирайте функцията `smooth_heatmap(heatmap)`.

In [ ]:
@torch.no_grad()
def smooth_heatmap(heatmap: torch.Tensor) -> torch.Tensor:
    # TODO: Реализирайте функцията тук.
    return heatmap


## Оценка на изображение (alarm) - опционално

Имплементирайте функцията `compute_alarm(heatmap)`.

In [ ]:
@torch.no_grad()
def compute_alarm(heatmap: torch.Tensor) -> float:
    """
    Това е функцията по подразбиране, която връща максимум.
    """
    return float(heatmap.max().item())

## Калибриране на прагове

Имплементирайте функцията `calibrate_thresholds(calibration_data)`.

**Вход:**
- `calibration_data`: списък от записи, всеки съдържа `X`, `atlas`, `atlas_id`, `shift_id`

**Изход:**
- речник от вида `thresholds[atlas_id][shift_id] = {"tau_img": ..., "tau_pix": ...}` - калибрационни прагове за `image-level` и за `pixel_level`

In [ ]:
@torch.no_grad()
def calibrate_thresholds(calibration_data):
    # calibration_data: списък от записи (X, atlas, atlas_id, shift_id)
    img_scores = {}   # (atlas_id, shift_id) -> list[float]
    pix_scores = {}   # (atlas_id, shift_id) -> list[np.ndarray]

    for item in calibration_data:
        X = item["X"]
        atlas = item["atlas"]
        atlas_id = int(item["atlas_id"])
        shift_id = int(item["shift_id"])
        key = (atlas_id, shift_id)

        # 1) heatmap за нормален пример
        heat = compute_heatmap(X, atlas)

        # 2) изглаждане
        heat = smooth_heatmap(heat)

        # 3) image-level оценка
        alarm = compute_alarm(heat)

        # 4) събираме данни (за по-късно смятане на прагове)
        img_scores.setdefault(key, []).append(float(alarm))
        pix_scores.setdefault(key, []).append(heat.reshape(-1).detach().cpu().numpy())

    # TODO: Тук сте вие
    thresholds = {}
    for (atlas_id, shift_id) in img_scores.keys():
        thresholds.setdefault(atlas_id, {})[shift_id] = {
            "tau_img": None,
            "tau_pix": None,
        }

    return thresholds


## Тестване

Използвайте предоставените валидационни данни, за да проверите решението си.

In [ ]:
# --- Оценяване върху val_data ---

def auc_safe(y_true, y_score):
    """ROC AUC, но ако няма и двата класа -> NaN."""
    y_true = np.asarray(y_true, dtype=np.uint8)
    y_score = np.asarray(y_score, dtype=np.float32)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

thresholds = calibrate_thresholds(calibration_data)  # може да е празно/None — OK за AUC

img_true = []
img_score = []

pix_true_all = []
pix_score_all = []

for m in manifest:
    data = torch.load(m["file"], map_location="cpu")

    atlas_id = int(m["atlas_id"])
    shift_id = int(m.get("shift_id", 0))
    X = data["X"]
    atlas = atlases[atlas_id]

    # heatmap + smoothing
    heat = compute_heatmap(X, atlas)
    heat = smooth_heatmap(heat)

    # Raw scores
    raw_alarm = float(compute_alarm(heat))
    raw_heat = heat.detach().cpu().view(-1).numpy().astype(np.float32)

    # Normalize if thresholds exist
    t = thresholds.get(atlas_id, {}).get(shift_id, None)
    tau_img = t.get("tau_img", 1.0) if t else 1.0
    tau_pix = t.get("tau_pix", 1.0) if t else 1.0
    
    if tau_img is None or tau_img <= 1e-12: tau_img = 1.0
    if tau_pix is None or tau_pix <= 1e-12: tau_pix = 1.0

    # image-level ground truth и score
    y_img = 1 if bool(data.get("is_anomaly", False)) else 0
    img_true.append(y_img)
    img_score.append(raw_alarm / tau_img)

    # heatmap-level ground truth и score
    gt = data["mask"].to(torch.uint8).view(-1).numpy()  # (1369,)
    pix_true_all.append(gt)
    pix_score_all.append(raw_heat / tau_pix)

# Метрики: AUC и финален резултат
img_auc = auc_safe(img_true, img_score)

pix_true = np.concatenate(pix_true_all, axis=0).astype(np.uint8)
pix_score = np.concatenate(pix_score_all, axis=0).astype(np.float32)
heat_auc = auc_safe(pix_true, pix_score)

final_score = 0.5 * img_auc + 0.5 * heat_auc

print(f"img_auc      : {img_auc:.4f}")
print(f"heatmap_auc  : {heat_auc:.4f}")
print(f"final_score  : {final_score:.4f}")
print(f"val_images   : {len(img_true)} (anom={sum(img_true)}, normal={len(img_true)-sum(img_true)})")
